# Fase 3: Diseño algorítmico, Complejidad y Programación Orientada a Objetos
**Proyecto:** Análisis de datos de amenazas de ciberseguridad  
**Integrantes:** Jennifer Nilo – Patricio Núñez

---

## 1. Exploración e Ingesta Defensiva
Iniciamos configurando el entorno e importando los datos. Si el archivo masivo no está disponible (por exclusión de GitHub), el sistema cargará automáticamente nuestra muestra cruda para garantizar la reproducibilidad. Por ello, en esta fase integraremos el preprocesamiento de la Fase 2 refactorizado hacia un paradigma de Programación Orientada a Objetos (POO).

In [ ]:
import pandas as pd
import numpy as np
import timeit
import sys
import os

# Fijamos la semilla de aleatoriedad para garantizar reproducibilidad global
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Aumentamos el límite de recursión por seguridad en nuestro algoritmo Merge Sort
sys.setrecursionlimit(5000)

# Vinculamos nuestra arquitectura modular local (/src/) al path de Python
ruta_src = os.path.abspath('./src')
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

def cargar_datos_crudos():
    """Carga el CSV crudo. Si no existe el masivo, usa la muestra reproducible."""
    ruta_raw = '../data/raw/cybersecurity_threat_detection_logs.csv'
    ruta_muestra = '../data/raw/cybersecurity_threat_detection_logs_muestra_500000.csv'

    if os.path.exists(ruta_raw):
        print(f"Cargando dataset masivo local...")
        df = pd.read_csv(ruta_raw)
    elif os.path.exists(ruta_muestra):
        print(f"[MODO EVALUACIÓN] Dataset masivo no detectado. Cargando muestra reproducible...")
        df = pd.read_csv(ruta_muestra)
    else:
        raise FileNotFoundError("No se encontró ningún dataset en data/raw/.")
    
    print(f"Datos cargados: {df.shape[0]:,} filas, {df.shape[1]} columnas")
    return df

# 1) Cargamos los datos crudos iniciales
df_crudo = cargar_datos_crudos()

## 2. Herencia, Polimorfismo y Encapsulamiento (POO)
En lugar de tener funciones sueltas, hemos agrupado los datos y las acciones en **Clases** dentro del archivo `src/preprocesamiento.py`. 

Aplicamos los pilares de la POO mediante el patrón *Strategy*:
* **Herencia:** Creamos una clase base `Transformador` de la cual heredan especializaciones como `LimpiadorNulos` o `EscaladorMinMax`.
* **Polimorfismo:** El orquestador no necesita saber qué hace cada transformación, solo llama al método `.aplicar()` en todas ellas por igual.
* **Encapsulamiento:** La clase `PipelinePreprocesamiento` protege el *DataFrame* operando sobre una copia interna.

In [ ]:
# Importamos nuestras clases de F3/src/preprocesamiento.py
from preprocesamiento import (
    PipelinePreprocesamiento, 
    LimpiadorNulos, 
    FiltroIPs, 
    TransformadorFechas, 
    EscaladorMinMax, 
    IngenieriaCaracteristicas
)

# 2) Armamos el pipeline
pipeline_pre = PipelinePreprocesamiento([
    LimpiadorNulos(),
    FiltroIPs(),
    TransformadorFechas(),
    EscaladorMinMax(),
    IngenieriaCaracteristicas()
])

# 3) Ejecutamos el pipeline completo en una sola línea
print("Ejecutando transformaciones orientadas a objetos...")
df_procesado = pipeline_pre.ejecutar(df_crudo)

# 4) Validamos que no queden nulos ni texto
nulos_totales = df_procesado.isnull().sum().sum()
texto_restante = df_procesado.select_dtypes(include=['object', 'string']).columns.tolist()

assert nulos_totales == 0, "Error: Quedan valores nulos."
assert not texto_restante, "Error: Quedan columnas sin codificar."

print(f"Dimensiones finales: {df_procesado.shape}")
display(df_procesado.head(3))

## 3. Recursividad en el Núcleo Algorítmico
Una función **recursiva** se llama a sí misma dividiendo un problema grande en partes más pequeñas hasta llegar a un caso base. 

En nuestro módulo `src/motor_analisis.py`, implementamos la clase `AnalizadorAlertas`, la cual utiliza recursividad pura mediante el algoritmo *Merge Sort* para ordenar el volumen de red (`bytes_escalados`). Validaremos que entregue el mismo resultado matemático que el enfoque iterativo usando un bloque `assert`.

In [ ]:
# Importamos la clase algorítmica de F3/src/motor_analisis.py
from motor_analisis import AnalizadorAlertas

# Transformamos la columna a lista nativa para medir rendimiento matemático puro
lista_bytes = df_procesado['bytes_escalados'].tolist()

# 5) Instanciamos nuestro motor algorítmico (Encapsulamiento)
motor = AnalizadorAlertas(lista_bytes)

# 6) Validación técnica de unidad
muestra_control = motor.lista_bytes[:10]
resultado_iter = motor.ordenar_iterativo(muestra_control)
resultado_rec = motor.ordenar_recursivo(muestra_control)

# Si hay error en la recursividad, este assert detiene la ejecución
assert resultado_iter == resultado_rec, "Error: Los algoritmos producen resultados diferentes."
print("Validación exitosa: El algoritmo iterativo y recursivo son matemáticamente idénticos.")

## 4. Eficiencia Algorítmica: 
Cuando los datos de ciberseguridad crecen a millones de eventos, la complejidad computacional es el factor crítico. Utilizaremos la librería `timeit` para capturar los tiempos y `matplotlib` para la visualización del rendimiento:

1. **Selection Sort (Iterativo - $O(n^2)$):** Lo probaremos con una muestra segura, ya que su ineficiencia matemática colapsaría la memoria con la carga masiva.
2. **Merge Sort (Recursivo - $O(n \log n)$):** Lo ejecutaremos sobre el *dataset* completo para demostrar su superioridad y escalabilidad real.

In [ ]:
import matplotlib.pyplot as plt

print("--- COMPARATIVA DE COMPLEJIDAD ASINTÓTICA ---\n")

total_registros = len(motor.lista_bytes)
muestra_segura = min(10000, total_registros) 

# Medición 1: Iterativo O(n^2)
print(f"[1] Midiendo Selection Sort O(n^2) con muestra de {muestra_segura:,} registros...")
t_iterativo = timeit.timeit(lambda: motor.ordenar_iterativo(motor.lista_bytes[:muestra_segura]), number=1)
print(f"    -> Tiempo parcial: {t_iterativo:.4f} segundos")

# Medición 2: Recursivo (Divide y Vencerás O(n log n))
print(f"\n[2] Midiendo Merge Sort O(n log n) con TODOS los {total_registros:,} registros...")
t_recursivo = timeit.timeit(lambda: motor.ordenar_recursivo(motor.lista_bytes), number=1)
print(f"    -> Tiempo total: {t_recursivo:.4f} segundos")

# Proyección Matemática para O(n^2)
factor = total_registros / muestra_segura
tiempo_iterativo_total_proyectado = t_iterativo * (factor ** 2)
proyeccion_iterativo_dias = tiempo_iterativo_total_proyectado / 86400

print("\n--- GENERANDO VISUALIZACIÓN DE RENDIMIENTO ---")

# --- GRÁFICO MATPLOTLIB ---
nombres_algoritmos = ['Selection Sort $O(n^2)$\n(Proyectado a 6M)', 'Merge Sort $O(n \\log n)$\n(Real sobre 6M)']
tiempos_segundos = [tiempo_iterativo_total_proyectado, t_recursivo]

plt.figure(figsize=(10, 6))
# Usamos rojo para el ineficiente y verde para el eficiente
barras = plt.bar(nombres_algoritmos, tiempos_segundos, color=['#e74c3c', '#2ecc71'], edgecolor='black')

# Escala logarítmica es obligatoria aquí por la diferencia abismal entre segundos y días
plt.yscale('log')
plt.ylabel('Tiempo de procesamiento en Segundos (Escala Logarítmica)', fontsize=12)
plt.title('Comparativa de Rendimiento Algorítmico: Triaje de 6.000.000 de Eventos', fontsize=14, pad=15)

# Añadimos las etiquetas de texto encima de las barras para fácil lectura
plt.text(0, tiempo_iterativo_total_proyectado * 1.2, f'{proyeccion_iterativo_dias:.1f} Días', ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.text(1, t_recursivo * 1.2, f'{t_recursivo:.2f} Segundos', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()